## Outfit Logic Foundation

### Project taxonomy

5 categories, matching the subCategory values in `combined_df.csv` and the label space of the trained ResNet50.

In [1]:
# Mirrors classifier_training.ipynb:SUBCATEGORY_TO_LABEL exactly — do not drift.
PROJECT_CATEGORIES = ["Tops", "Bottomwear", "Shoes", "Dress", "Outerwear"]

SUBCATEGORY_TO_LABEL = {
    "Tops":       0,
    "Bottomwear": 1,
    "Shoes":      2,
    "Dress":      3,
    "Outerwear":  4,
}
LABEL_TO_SUBCATEGORY = {v: k for k, v in SUBCATEGORY_TO_LABEL.items()}

# styles.csv baseColour values are mixed-case and include compound names ("Navy Blue").
# Normalize to a small palette the genre templates and color-harmony table use.
COLOR_NORMALIZE = {
    "Black": "black", "White": "white", "Grey": "gray", "Gray": "gray",
    "Blue": "blue", "Navy Blue": "blue", "Teal": "blue", "Turquoise Blue": "blue",
    "Red": "red", "Maroon": "red", "Burgundy": "red", "Rust": "red",
    "Green": "green", "Olive": "green", "Sea Green": "green", "Lime Green": "green",
    "Beige": "beige", "Cream": "beige", "Khaki": "beige", "Tan": "beige",
    "Brown": "brown", "Coffee Brown": "brown", "Mushroom Brown": "brown",
    "Pink": "pink", "Magenta": "pink", "Rose": "pink", "Peach": "pink",
    "Yellow": "yellow", "Mustard": "yellow", "Gold": "yellow",
    "Purple": "purple", "Lavender": "purple", "Mauve": "purple",
    "Orange": "orange", "Bronze": "orange",
}

def normalize_color(raw):
    if raw is None:
        return None
    return COLOR_NORMALIZE.get(str(raw).strip(), None)

### Wardrobe item schema

Canonical record for every item in the digital closet. Backend, UI, and recommendation engine all serialize through this.

In [2]:
from dataclasses import dataclass, field, asdict
from typing import Optional

CATEGORIES = set(PROJECT_CATEGORIES)
COLORS = {"black", "white", "gray", "blue", "red", "green",
          "beige", "brown", "pink", "yellow", "purple", "orange"}
GENRES = {"casual", "business_casual", "athletic", "punk", "minimalist"}


@dataclass
class WardrobeItem:
    item_id: str
    image_path: str
    category: str                    # one of PROJECT_CATEGORIES
    article_type: Optional[str] = None   # styles.csv articleType (Tshirts, Heels, Jeans, ...)
    color: Optional[str] = None
    style_tags: list = field(default_factory=list)
    season: str = "all_season"
    usage: Optional[str] = None      # styles.csv usage (Casual, Formal, Sports, Party)
    classifier_confidence: Optional[float] = None
    user_corrected: bool = False

    def to_dict(self):
        return asdict(self)


def validate(item: WardrobeItem):
    errors = []
    if item.category not in CATEGORIES:
        errors.append(f"category {item.category!r} not in {CATEGORIES}")
    if item.color is not None and item.color not in COLORS:
        errors.append(f"color {item.color!r} not in {COLORS}")
    bad_tags = [t for t in item.style_tags if t not in GENRES]
    if bad_tags:
        errors.append(f"unknown style_tags: {bad_tags}")
    return errors

### Genre templates

5 starter genres. Each defines required vs optional categories, preferred and forbidden colors, and the style tag a matching item should carry.

In [3]:
NEUTRALS = {"black", "white", "gray", "beige", "brown"}

GENRE_TEMPLATES = {
    "casual": {
        "required": ["Tops", "Bottomwear"],
        "optional": ["Outerwear", "Shoes"],
        "dress_alternative": ["Dress"],
        "preferred_colors": NEUTRALS | {"blue", "green"},
        "forbidden_colors": set(),
        "style_tag_required": "casual",
    },
    "business_casual": {
        "required": ["Tops", "Bottomwear"],
        "optional": ["Outerwear", "Shoes"],
        "dress_alternative": ["Dress", "Outerwear"],
        "preferred_colors": NEUTRALS | {"blue"},
        "forbidden_colors": {"red", "pink", "yellow", "purple", "orange"},
        "style_tag_required": "business_casual",
    },
    "athletic": {
        "required": ["Tops", "Bottomwear"],
        "optional": ["Shoes"],
        "dress_alternative": [],
        "preferred_colors": NEUTRALS | {"blue", "red"},
        "forbidden_colors": set(),
        "style_tag_required": "athletic",
    },
    "punk": {
        "required": ["Tops", "Bottomwear"],
        "optional": ["Outerwear", "Shoes"],
        "dress_alternative": ["Dress", "Outerwear"],
        "preferred_colors": {"black", "gray", "white", "red"},
        "forbidden_colors": {"beige", "pink", "yellow"},
        "style_tag_required": "punk",
    },
    "minimalist": {
        "required": ["Tops", "Bottomwear"],
        "optional": ["Outerwear", "Shoes"],
        "dress_alternative": ["Dress"],
        "preferred_colors": NEUTRALS,
        "forbidden_colors": {"red", "pink", "yellow", "purple", "green", "orange"},
        "style_tag_required": "minimalist",
    },
}

### Scoring

`score = category_completeness + genre_alignment + color_compatibility + layering_bonus`

Components stay separate so the UI can render an explainable "why this outfit works" string.

In [4]:
COLOR_HARMONY = {
    frozenset({"black", "white"}): 1.0,
    frozenset({"black", "gray"}): 1.0,
    frozenset({"white", "beige"}): 1.0,
    frozenset({"blue", "white"}): 0.9,
    frozenset({"black", "red"}): 0.8,
    frozenset({"beige", "brown"}): 0.9,
    frozenset({"gray", "blue"}): 0.8,
}


def category_completeness(items, genre):
    template = GENRE_TEMPLATES[genre]
    required = set(template["required"])
    present = {item.category for item in items}
    if template["dress_alternative"] and "Dress" in present:
        return 1.0
    if not required:
        return 1.0
    return len(required & present) / len(required)


def genre_alignment(items, genre):
    if not items:
        return 0.0
    return sum(1 for item in items if genre in item.style_tags) / len(items)


def color_compatibility(items, genre):
    template = GENRE_TEMPLATES[genre]
    preferred = template["preferred_colors"]
    forbidden = template["forbidden_colors"]
    colors = [item.color for item in items if item.color]
    if not colors:
        return 0.5

    in_palette = sum(1 for c in colors if c in preferred) / len(colors)
    forbidden_hits = sum(1 for c in colors if c in forbidden)

    pair_bonus = 0.0
    color_set = set(colors)
    for pair, score in COLOR_HARMONY.items():
        if pair.issubset(color_set):
            pair_bonus = max(pair_bonus, score)

    return max(0.0, in_palette + 0.2 * pair_bonus - 0.4 * forbidden_hits)


def layering_bonus(items):
    cats = {item.category for item in items}
    return 0.2 if "Outerwear" in cats and "Tops" in cats else 0.0


def score_outfit(items, genre):
    components = {
        "category_completeness": category_completeness(items, genre),
        "genre_alignment": genre_alignment(items, genre),
        "color_compatibility": color_compatibility(items, genre),
        "layering_bonus": layering_bonus(items),
    }
    components["total"] = sum(components.values())
    return components


def explain(items, genre):
    colors = sorted({item.color for item in items if item.color})
    cats = sorted({item.category for item in items})
    return (
        f"{genre.replace('_', ' ').title()} look: "
        f"{', '.join(cats)} in {', '.join(colors) if colors else 'mixed'} palette."
    )

### Outfit engine

`generate_outfits(wardrobe, genre)` filters by hard constraints (forbidden colors), groups by category, generates required + optional combos plus the dress fallback, scores them, and returns the top N. Falls back to a "what to upload next" message when no combo is possible.

In [5]:
from itertools import product


def filter_by_genre(wardrobe, genre):
    """Drop only hard-incompatible items (forbidden colors). Soft genre matching lives in the scorer."""
    template = GENRE_TEMPLATES[genre]
    forbidden = template["forbidden_colors"]
    return [item for item in wardrobe if item.color not in forbidden]


def group_by_category(items):
    groups = {}
    for item in items:
        groups.setdefault(item.category, []).append(item)
    return groups


def candidate_combos(groups, genre):
    template = GENRE_TEMPLATES[genre]
    required = template["required"]
    optional = template["optional"]
    dress_alt = template["dress_alternative"]

    combos = []

    if all(c in groups for c in required):
        required_lists = [groups[c] for c in required]
        for combo in product(*required_lists):
            combos.append(list(combo))
            for opt_cat in optional:
                if opt_cat in groups:
                    for opt_item in groups[opt_cat]:
                        combos.append([*combo, opt_item])

    if "Dress" in dress_alt and "Dress" in groups:
        for dress in groups["Dress"]:
            combos.append([dress])
            if "Outerwear" in groups:
                for outer in groups["Outerwear"]:
                    combos.append([dress, outer])
            if "Shoes" in groups:
                for shoe in groups["Shoes"]:
                    combos.append([dress, shoe])

    return combos


def missing_items_message(groups, genre):
    template = GENRE_TEMPLATES[genre]
    missing = [c for c in template["required"] if c not in groups]
    if missing:
        return f"Need to upload at least one: {', '.join(missing)}."
    return "No items match this genre yet."


def generate_outfits(wardrobe, genre, top_n=5):
    if genre not in GENRE_TEMPLATES:
        raise ValueError(f"unknown genre: {genre}")

    candidates = filter_by_genre(wardrobe, genre)
    groups = group_by_category(candidates)
    combos = candidate_combos(groups, genre)

    scored = []
    for combo in combos:
        scores = score_outfit(combo, genre)
        scored.append({
            "items": combo,
            "scores": scores,
            "explanation": explain(combo, genre),
        })

    scored.sort(key=lambda x: x["scores"]["total"], reverse=True)

    if not scored:
        return [{
            "items": [],
            "scores": {"total": 0.0},
            "explanation": missing_items_message(groups, genre),
        }]

    return scored[:top_n]

## Data Inventory & Train / Val / Test Split

Build the modeling dataset directly from `data/styles.csv` + `data/images.csv` (real image URLs) so this notebook does not depend on a prebuilt artifact. We merge on product id (`styles.id` vs `images.filename` stem), map raw labels to the project's 5-category taxonomy, then create a reproducible **80 / 10 / 10 stratified split** for W4 evaluation and W5 smoke tests.

In [6]:
import os
import pandas as pd
import numpy as np

styles_df = pd.read_csv("../data/styles.csv")
images_df = pd.read_csv("../data/images.csv")

# images.csv stores id inside filename like "15970.jpg".
images_df["file_id"] = images_df["filename"].astype(str).str.split(".").str[0].astype(int)

raw_df = styles_df.merge(
    images_df[["file_id", "link", "filename"]],
    left_on="id",
    right_on="file_id",
    how="inner",
)

# Mirror data_exploration.ipynb preprocessing logic.
# 1) Keep the same coarse label scope first.
combined_df = raw_df[raw_df["subCategory"].isin(["Topwear", "Bottomwear", "Shoes", "Dress"])].copy()

# 2) Align usage cleanup/filter used in exploration.
combined_df["usage"] = combined_df["usage"].replace("Smart Casual", "Casual")
combined_df = combined_df[combined_df["usage"].isin(["Casual", "Formal", "Sports", "Party"])].copy()

# 3) Dresses override: some rows are topwear but articleType is Dresses.
combined_df["subCategory"] = combined_df.apply(
    lambda row: "Dress" if row["articleType"] == "Dresses" else row["subCategory"],
    axis=1,
)

# 4) Remove article types excluded in exploration notebook.
EXCLUDED_ARTICLE_TYPES = {"Kurtas", "Kurtis", "Belts", "Suspenders"}
combined_df = combined_df[~combined_df["articleType"].isin(EXCLUDED_ARTICLE_TYPES)].copy()

# 5) Split Topwear into Tops vs Outerwear.
TOPS_ARTICLES = {
    "Shirts", "Tshirts", "Tops", "Tunics", "Tees", "Sweatshirts", "Sweaters",
    "Kurtis", "Kurtas", "Blouses", "Dupatta", "Nehru Jackets", "Rompers", "Suits",
}
OUTERWEAR_ARTICLES = {
    "Blazers", "Jackets", "Waistcoat", "Shrug", "Cardigans", "Coats", "Rain Jacket",
}


def remap_subcategory(row):
    sub = row["subCategory"]
    article = row["articleType"]

    if sub == "Topwear":
        if article in OUTERWEAR_ARTICLES:
            return "Outerwear"
        return "Tops"  # default bucket for other Topwear article types

    simple_map = {
        "Bottomwear": "Bottomwear",
        "Shoes": "Shoes",
        "Dress": "Dress",
    }
    return simple_map.get(sub)


combined_df["subCategory"] = combined_df.apply(remap_subcategory, axis=1)
combined_df = combined_df[combined_df["subCategory"].isin(PROJECT_CATEGORIES)].reset_index(drop=True)

print(f"styles rows: {len(styles_df):,}")
print(f"images rows: {len(images_df):,}")
print(f"merged rows: {len(raw_df):,}")
print(f"project rows after mapped filtering: {len(combined_df):,}")
print()
print("class distribution (subCategory):")
counts = combined_df["subCategory"].value_counts()
for cat in PROJECT_CATEGORIES:
    print(f"  {cat:<11} {counts.get(cat, 0):>6,}")

# Stratified 80/10/10 split, seed=42 so the team can reproduce.
rng = np.random.default_rng(seed=42)
train_rows, val_rows, test_rows = [], [], []
for cat, group in combined_df.groupby("subCategory"):
    idx = rng.permutation(len(group))
    n = len(group)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)
    rows = group.iloc[idx]
    train_rows.append(rows.iloc[:n_train])
    val_rows.append(rows.iloc[n_train:n_train + n_val])
    test_rows.append(rows.iloc[n_train + n_val:])

train_df = pd.concat(train_rows).sample(frac=1, random_state=42).reset_index(drop=True)
val_df = pd.concat(val_rows).sample(frac=1, random_state=42).reset_index(drop=True)
test_df = pd.concat(test_rows).sample(frac=1, random_state=42).reset_index(drop=True)

print()
print(f"split sizes: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
print()
print(f"{'category':<11} {'train':>8} {'val':>8} {'test':>8} {'total':>8}")
for cat in PROJECT_CATEGORIES:
    tr = (train_df["subCategory"] == cat).sum()
    va = (val_df["subCategory"] == cat).sum()
    te = (test_df["subCategory"] == cat).sum()
    print(f"  {cat:<9} {tr:>8,} {va:>8,} {te:>8,} {tr + va + te:>8,}")

# Persist splits for reuse in training/eval notebooks.
os.makedirs("../data/splits", exist_ok=True)
train_df.to_csv("../data/splits/train_df.csv", index=False)
val_df.to_csv("../data/splits/val_df.csv", index=False)
test_df.to_csv("../data/splits/test_df.csv", index=False)
print()
print("wrote splits to data/splits/{train,val,test}_df.csv")

styles rows: 44,446
images rows: 44,446
merged rows: 44,446
project rows after mapped filtering: 23,392

class distribution (subCategory):
  Tops        12,763
  Bottomwear   2,537
  Shoes        7,321
  Dress          480
  Outerwear      291

split sizes: train=18,711  val=2,338  test=2,343

category       train      val     test    total
  Tops        10,210    1,276    1,277   12,763
  Bottomwear    2,029      253      255    2,537
  Shoes        5,856      732      733    7,321
  Dress          384       48       48      480
  Outerwear      232       29       30      291

wrote splits to data/splits/{train,val,test}_df.csv


## Classifier Evaluation

### Pure-python eval harness

Operates on `(y_true, y_pred)` lists. Returns accuracy, per-class P/R/F1, the confusion matrix, and a list of low-F1 classes that should be relabeled or get more data.

In [7]:
from collections import defaultdict


def evaluate(y_true, y_pred, classes):
    if len(y_true) != len(y_pred):
        raise ValueError("y_true and y_pred must have same length")

    n = len(y_true)
    acc = sum(t == p for t, p in zip(y_true, y_pred)) / n if n else 0.0

    cm = defaultdict(int)
    for t, p in zip(y_true, y_pred):
        cm[(t, p)] += 1

    per_class = {}
    for cls in classes:
        tp = cm.get((cls, cls), 0)
        fp = sum(v for (t, p), v in cm.items() if p == cls and t != cls)
        fn = sum(v for (t, p), v in cm.items() if t == cls and p != cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        per_class[cls] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": tp + fn,
        }

    # biggest misses first so we can quickly inspect failure modes
    errors = [(t, p, v) for (t, p), v in cm.items() if t != p]
    errors.sort(key=lambda x: x[2], reverse=True)

    return {
        "accuracy": acc,
        "per_class": per_class,
        "confusion": dict(cm),
        "top_errors": errors[:20],
    }


def summary_table(result):
    lines = [f"overall accuracy: {result['accuracy']:.4f}", ""]
    lines.append(f"{'class':<12} {'precision':>10} {'recall':>10} {'f1':>10} {'support':>10}")

    for cls, stats in sorted(result["per_class"].items()):
        lines.append(
            f"{cls:<12} {stats['precision']:>10.4f} {stats['recall']:>10.4f} "
            f"{stats['f1']:>10.4f} {stats['support']:>10}"
        )

    lines.append("")
    lines.append("top confusions (true -> pred, count):")
    for true_cls, pred_cls, count in result["top_errors"][:10]:
        lines.append(f"  {true_cls} -> {pred_cls}: {count}")

    return "\n".join(lines)


def relabel_recommendations(result, threshold=0.5):
    notes = []

    for cls, stats in result["per_class"].items():
        if stats["support"] == 0 or stats["f1"] >= threshold:
            continue

        confused_as = [
            (pred, cnt)
            for (true, pred), cnt in result["confusion"].items()
            if true == cls and pred != cls
        ]
        confused_as.sort(key=lambda x: x[1], reverse=True)

        if confused_as:
            worst_pred, count = confused_as[0]
            notes.append(
                f"class '{cls}' f1={stats['f1']:.2f} (support={stats['support']}); "
                f"most confused with '{worst_pred}' ({count}x). "
                "consider: more data, merge, or relabel."
            )

    return notes

### Run the saved ResNet50 against the test split

Loads `models/resnet50_stylesync.pt` and scores it on `data/splits/test_df.csv`. If the model file doesn't exist yet, the cell skips gracefully — re-run after `classifier_training.ipynb` saves the weights.

Test images are downloaded from the `link` column in parallel and cached to `data/img_cache/` so re-runs are instant. (The `data/img/` folder is unrelated DeepFashion data and isn't used.) Tune `MAX_TEST_IMAGES` to control runtime.

In [8]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests

MODEL_PATH = "../models/resnet50_stylesync.pt"
IMAGE_CACHE_DIR = "../data/img_cache"   # cached after first run; re-runs are instant
MAX_TEST_IMAGES = 200                    # bump for a full eval pass
DOWNLOAD_WORKERS = 16                    # parallel HTTP fetches

os.makedirs(IMAGE_CACHE_DIR, exist_ok=True)

if not os.path.exists(MODEL_PATH):
    print(f"skipping: {MODEL_PATH} not found. "
          f"Re-run classifier_training.ipynb to save the model, then re-execute this cell.")
else:
    import torch
    import torch.nn as nn
    from torchvision import transforms, models
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image

    test_subset = test_df.head(MAX_TEST_IMAGES).reset_index(drop=True)

    def cache_path(image_id):
        return os.path.join(IMAGE_CACHE_DIR, f"{image_id}.jpg")

    def fetch_one(row):
        path = cache_path(row["id"])
        if os.path.exists(path):
            return row["id"], True
        try:
            resp = requests.get(row["link"], timeout=10)
            resp.raise_for_status()
            with open(path, "wb") as f:
                f.write(resp.content)
            return row["id"], True
        except Exception:
            return row["id"], False

    rows = list(test_subset.to_dict("records"))
    needed = [r for r in rows if not os.path.exists(cache_path(r["id"]))]
    print(f"images needed: {len(rows)}  cached: {len(rows) - len(needed)}  to fetch: {len(needed)}")

    if needed:
        ok_count = 0
        with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
            futures = [ex.submit(fetch_one, r) for r in needed]
            for i, fut in enumerate(as_completed(futures), 1):
                _, ok = fut.result()
                if ok:
                    ok_count += 1
                if i % 50 == 0:
                    print(f"  fetched {i}/{len(needed)}")
        print(f"  done. {ok_count}/{len(needed)} succeeded")

    NUM_CLASSES = 5
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model = model.to(device).eval()

    tfm = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    class TestDataset(Dataset):
        def __init__(self, df):
            self.df = df.reset_index(drop=True)
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            label = SUBCATEGORY_TO_LABEL[row["subCategory"]]
            path = cache_path(row["id"])
            try:
                img = Image.open(path).convert("RGB")
                return tfm(img), label, True
            except Exception:
                return torch.zeros(3, 224, 224), label, False

    loader = DataLoader(TestDataset(test_subset), batch_size=32, shuffle=False, num_workers=0)

    y_true, y_pred = [], []
    skipped = 0
    with torch.no_grad():
        for imgs, labels, ok in loader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(dim=1).cpu().numpy()
            for p, l, valid in zip(preds, labels.numpy(), ok.numpy()):
                if not valid:
                    skipped += 1
                    continue
                y_true.append(LABEL_TO_SUBCATEGORY[int(l)])
                y_pred.append(LABEL_TO_SUBCATEGORY[int(p)])

    print()
    print(f"evaluated on {len(y_true):,} images ({skipped} skipped due to load failure)")
    print()
    result = evaluate(y_true, y_pred, classes=PROJECT_CATEGORIES)
    print(summary_table(result))
    print()
    for s in relabel_recommendations(result, threshold=0.7):
        print("RELABEL:", s)

images needed: 200  cached: 26  to fetch: 174
  fetched 50/174
  fetched 100/174
  fetched 150/174
  done. 174/174 succeeded

evaluated on 200 images (0 skipped due to load failure)

overall accuracy: 0.8700

class         precision     recall         f1    support
Bottomwear       1.0000     0.1818     0.3077         22
Dress            0.0000     0.0000     0.0000          6
Outerwear        0.0000     0.0000     0.0000          2
Shoes            1.0000     1.0000     1.0000         60
Tops             0.8088     1.0000     0.8943        110

top confusions (true -> pred, count):
  Bottomwear -> Tops: 18
  Dress -> Tops: 6
  Outerwear -> Tops: 2

RELABEL: class 'Bottomwear' f1=0.31 (support=22); most confused with 'Tops' (18x). consider: more data, merge, or relabel.
RELABEL: class 'Dress' f1=0.00 (support=6); most confused with 'Tops' (6x). consider: more data, merge, or relabel.
RELABEL: class 'Outerwear' f1=0.00 (support=2); most confused with 'Tops' (2x). consider: more data, me

## Style Tags + Metadata Pipeline

### Style tag rules

Hybrid mapping: `usage` + `articleType` + `baseColour` from styles.csv → project genres. We have rich product metadata so we don't need DeepFashion's 1000-attribute vocabulary; the structured columns are cleaner signals.

In [9]:
# usage column directly maps to a starter genre
USAGE_TO_GENRES = {
    "Casual": ["casual"],
    "Sports": ["athletic"],
    "Formal": ["business_casual"],
    "Party": ["punk"],          # weakest signal — refine after user feedback
    "Travel": ["casual"],
}

# articleType is more granular than subCategory and disambiguates ambiguous items.
# E.g. Tshirts -> casual, Shirts -> business_casual / casual, Heels -> business_casual.
ARTICLE_TYPE_TO_GENRES = {
    # tops
    "Tshirts": ["casual"], "Shirts": ["business_casual", "casual"],
    "Tops": ["casual"], "Tunics": ["casual"], "Sweatshirts": ["casual", "athletic"],
    "Sweaters": ["casual", "minimalist"],
    # bottoms
    "Jeans": ["casual"], "Trousers": ["business_casual"],
    "Shorts": ["casual"], "Track Pants": ["athletic"],
    "Leggings": ["athletic"], "Capris": ["casual"], "Skirts": ["casual"],
    # dresses
    "Dresses": ["casual", "business_casual"],
    # outerwear
    "Jackets": ["casual", "punk"], "Blazers": ["business_casual"],
    "Waistcoat": ["business_casual"],
    # shoes
    "Casual Shoes": ["casual"], "Sports Shoes": ["athletic"],
    "Formal Shoes": ["business_casual"], "Heels": ["business_casual"],
    "Flats": ["casual", "minimalist"], "Sandals": ["casual"],
    "Flip Flops": ["casual"],
}


def tags_from_usage(usage):
    if not usage:
        return []
    return USAGE_TO_GENRES.get(usage, [])


def tags_from_article_type(article_type):
    if not article_type:
        return []
    return ARTICLE_TYPE_TO_GENRES.get(article_type, [])


def tags_from_color(color, category):
    if color is None:
        return []
    if color == "black" and category in {"Outerwear", "Bottomwear"}:
        return ["punk", "minimalist"]
    if color in {"black", "white", "gray", "beige"}:
        return ["minimalist"]
    return []


def assign_style_tags(category, color, article_type=None, usage=None):
    tags = set()
    tags.update(tags_from_usage(usage))
    tags.update(tags_from_article_type(article_type))
    tags.update(tags_from_color(color, category))
    if not tags:
        tags.add("casual")  # conservative fallback
    return sorted(tags)

### Per-upload metadata pipeline

Orchestrates: `image -> classifier -> color extractor -> style tagger -> validated WardrobeItem`. The CV team owns `classify` (loads the saved ResNet50 and returns `(subCategory, confidence)`) and `extract_color` (returns a normalized color string). This layer just calls them.

In [10]:
import json
import uuid


def process_upload(image_path, classify, extract_color,
                   article_type=None, usage=None, confidence_threshold=0.5):
    """Run the full upload pipeline and return a validated WardrobeItem."""
    sub_category, confidence = classify(image_path)
    if sub_category not in CATEGORIES:
        raise ValueError(f"classifier returned unknown subCategory: {sub_category!r}")

    color = extract_color(image_path)

    style_tags = assign_style_tags(
        category=sub_category,
        color=color,
        article_type=article_type,
        usage=usage,
    )

    item = WardrobeItem(
        item_id=str(uuid.uuid4())[:8],
        image_path=image_path,
        category=sub_category,
        article_type=article_type,
        color=color,
        usage=usage,
        style_tags=style_tags,
        classifier_confidence=confidence,
        user_corrected=False,
    )

    errors = validate(item)
    if errors:
        raise ValueError(f"invalid wardrobe item: {errors}")
    return item


def apply_user_correction(item, new_category=None, new_color=None, new_style_tags=None):
    if new_category:
        item.category = new_category
    if new_color:
        item.color = new_color
    if new_style_tags is not None:
        item.style_tags = new_style_tags
    item.user_corrected = True
    return item


def save_wardrobe(items, path):
    with open(path, "w") as f:
        json.dump([item.to_dict() for item in items], f, indent=2)


def load_wardrobe(path):
    with open(path) as f:
        data = json.load(f)
    return [WardrobeItem(**row) for row in data]


def is_low_confidence(item, threshold=0.5):
    return (item.classifier_confidence is not None
            and item.classifier_confidence < threshold)

## Smoke test — real images from source CSVs

Pulls one real row per `articleType` from the merged `styles.csv` + `images.csv` dataset (`combined_df` built in the split cell), downloads each real product image via `link`, runs the saved ResNet50, then feeds the items into the outfit engine. This is an end-to-end exercise of W3 (outfit engine) + W4 (classifier eval wiring) + W5 (style tags + metadata) on real photos.

In [11]:
# Pick one real row per articleType so we get visually distinct items.
SAMPLE_ARTICLE_TYPES = [
    "Tshirts", "Shirts",
    "Trousers", "Track Pants",
    "Blazers", "Jackets",
    "Sports Shoes", "Formal Shoes",
    "Dresses",
]

sample_rows = []
for at in SAMPLE_ARTICLE_TYPES:
    matches = combined_df[combined_df["articleType"] == at]
    if len(matches) > 0:
        sample_rows.append(matches.sample(1, random_state=7).iloc[0])

# Fetch the images (reuses the cache populated by the W4 eval cell).
os.makedirs(IMAGE_CACHE_DIR, exist_ok=True)
fetched = []
for row in sample_rows:
    path = cache_path(row["id"])
    if not os.path.exists(path):
        try:
            r = requests.get(row["link"], timeout=10); r.raise_for_status()
            with open(path, "wb") as f: f.write(r.content)
        except Exception as e:
            print(f"  fetch failed for {row['productDisplayName']}: {e}")
            continue
    fetched.append((row, path))
print(f"got {len(fetched)} real product images")

# Run the saved classifier on each so the WardrobeItem.category is the predicted one.
import torch
from torchvision import transforms, models
from torch import nn
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clf = models.resnet50(weights=None)
clf.fc = nn.Linear(clf.fc.in_features, 5)
clf.load_state_dict(torch.load(MODEL_PATH, map_location=device))
clf = clf.to(device).eval()

tfm = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def classify_image(path):
    img = Image.open(path).convert("RGB")
    with torch.no_grad():
        logits = clf(tfm(img).unsqueeze(0).to(device))
        probs = torch.softmax(logits, dim=1)[0]
        idx = int(probs.argmax().item())
    return LABEL_TO_SUBCATEGORY[idx], float(probs[idx])

# Build a real wardrobe.
wardrobe = []
print()
print(f"{'#':<3} {'true':<11} {'pred':<11} {'conf':>6}  {'articleType':<14} {'color':<7}  product")
for i, (row, path) in enumerate(fetched, 1):
    pred_cat, conf = classify_image(path)
    color = normalize_color(row["baseColour"])
    style_tags = assign_style_tags(pred_cat, color, row["articleType"], row["usage"])

    wardrobe.append(WardrobeItem(
        item_id=str(row["id"]),
        image_path=path,
        category=pred_cat,
        article_type=row["articleType"],
        color=color,
        usage=row["usage"],
        style_tags=style_tags,
        classifier_confidence=conf,
    ))
    flag = "" if pred_cat == row["subCategory"] else "  <- mismatch"
    print(f"{i:<3} {row['subCategory']:<11} {pred_cat:<11} {conf:>6.2f}  "
          f"{row['articleType']:<14} {str(color):<7}  {row['productDisplayName'][:50]}{flag}")

print()
print("=== business_casual outfits ===")
for o in generate_outfits(wardrobe, "business_casual", top_n=3):
    items = [f"{i.article_type}({i.color})" for i in o["items"]]
    print(f"  {items} score={o['scores']['total']:.2f}  | {o['explanation']}")

print()
print("=== athletic outfits ===")
for o in generate_outfits(wardrobe, "athletic", top_n=3):
    items = [f"{i.article_type}({i.color})" for i in o["items"]]
    print(f"  {items} score={o['scores']['total']:.2f}  | {o['explanation']}")

got 9 real product images

#   true        pred          conf  articleType    color    product
1   Tops        Tops          0.79  Tshirts        white    Tokyo Talkies Women Printed White T-shirt
2   Tops        Tops          0.84  Shirts         purple   Scullers Men Scul Purple Shirts
3   Bottomwear  Tops          0.52  Trousers       blue     Indian Terrain Men Navy Blue Trousers  <- mismatch
4   Bottomwear  Bottomwear    0.52  Track Pants    black    Nike Women Black Track Pants
5   Outerwear   Tops          0.70  Blazers        black    Scullers For Her Black Blazer  <- mismatch
6   Outerwear   Tops          0.75  Jackets        pink     Forever New Women Soft Magnolia Pink Jacket  <- mismatch
7   Shoes       Shoes         0.83  Sports Shoes   gray     Puma Women Axis Grey Sports Shoes
8   Shoes       Shoes         0.76  Formal Shoes   black    Hush Puppies Men Hpo2 Flex Black Formal Shoes
9   Dress       Tops          0.79  Dresses        pink     Sepia Women Pink Dress  <- mism

In [12]:
# Style tag inference for a few items
print(assign_style_tags("Outerwear",  "black", "Blazers",      "Formal"))
print(assign_style_tags("Bottomwear", "gray",  "Track Pants",  "Sports"))
print(assign_style_tags("Tops",       "white", "Tshirts",      "Casual"))
print(assign_style_tags("Shoes",      "black", "Heels",        "Formal"))

['business_casual', 'minimalist', 'punk']
['athletic', 'minimalist']
['casual', 'minimalist']
['business_casual', 'minimalist']


In [13]:
# Evaluation harness 
y_true = ["Tops", "Tops", "Tops", "Bottomwear", "Bottomwear", "Dress", "Outerwear", "Outerwear", "Shoes", "Shoes"]
y_pred = ["Tops", "Tops", "Bottomwear", "Bottomwear", "Bottomwear", "Dress", "Outerwear", "Tops", "Shoes", "Shoes"]
result = evaluate(y_true, y_pred, classes=PROJECT_CATEGORIES)
print(summary_table(result))
print()
for s in relabel_recommendations(result, threshold=0.7):
    print("RELABEL:", s)

overall accuracy: 0.8000

class         precision     recall         f1    support
Bottomwear       0.6667     1.0000     0.8000          2
Dress            1.0000     1.0000     1.0000          1
Outerwear        1.0000     0.5000     0.6667          2
Shoes            1.0000     1.0000     1.0000          2
Tops             0.6667     0.6667     0.6667          3

top confusions (true -> pred, count):
  Tops -> Bottomwear: 1
  Outerwear -> Tops: 1

RELABEL: class 'Tops' f1=0.67 (support=3); most confused with 'Bottomwear' (1x). consider: more data, merge, or relabel.
RELABEL: class 'Outerwear' f1=0.67 (support=2); most confused with 'Tops' (1x). consider: more data, merge, or relabel.
